# EDA — Модуль 2: картирование гарей и степень поражения (BS)

Этот ноутбук — исследовательская часть отчёта к решению модуля BS (`unet_train.ipynb`,
`inference_new.py`). Задача — показать, **от каких наблюдений в данных мы пришли к
конкретной двухэтапной схеме** финального пайплайна: к признакам бустинга (dNBR, dNDVI,
dNBR2, радар, оконные статистики трёх масштабов, контекст второго прохода), к выбору
U-Net поверх бустинга и к её 40-канальному входу, к параметрам обучения (расписание на
60 эпох, остановка на 25-й) и к параметрам смеси (`unet_weight=0.45`, `threshold=0.4`).

Ноутбук покрывает только **модуль картирования гарей (BS)**. Активное горение (AF) —
отдельный модуль с собственным EDA (`EDA_active_fire.ipynb`), пересечений по каналам и
данным между модулями нет.

Порядок изложения тот же, что и в EDA по AF: **наблюдение → гипотеза → решение,
принятое в `unet_train.ipynb` / `inference_new.py`.** Так пункт критерия «EDA и
наблюдения» напрямую поддерживает пункт «Инженерные и предметные решения», а раздел
про baseline — пункт «Baseline и отправная точка» (раздел 2 критериев оценки кейса).

In [6]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# Пути — те же, что и в unet_train.ipynb / inference_new.py
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train" / "bs").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Сначала выполните prepare_data.py --scope train")
ROOT = PROJECT / "data" / "train" / "bs"
FIRMS_DIR = ROOT
MASKS = ROOT / "masks"
SIZE = 512
BANDS_S2 = ["B2", "B3", "B4", "B5", "B6", "B7", "B8A", "B11", "B12", "SCL"]

meta = pd.read_csv(ROOT / "meta.csv")
meta = meta[meta.kind == "bs"].reset_index(drop=True)

print(f"чипов BS в meta.csv: {len(meta)}")
meta.head()

чипов BS в meta.csv: 224


,chip_id,kind,fire_event_id,region,epsg,x_min,y_min,x_max,y_max,width,...,s1_date_pre,s1_date_post,valid_frac,cloud_frac,landcover_top,n_fire_px,burn_area_ha,sev1_px,sev2_px,sev3_px
0,BS_tr_000001,bs,FE00657,NaN,32638.0,491520.0,5376000.0,501760.0,5386240.0,512,...,2019-04-17,2019-04-29,0.9963,0.0037,NaN,NaN,454.80,11360.0,10.0,0.0
1,BS_tr_000002,bs,FE01044,NaN,32638.0,471040.0,5068800.0,481280.0,5079040.0,512,...,2019-06-04,2019-06-21,0.8195,0.1805,NaN,NaN,2865.92,40651.0,1699.0,29298.0
2,BS_tr_000003,bs,FE01122,NaN,32638.0,655360.0,5396480.0,665600.0,5406720.0,512,...,2019-06-16,2019-06-28,0.9921,0.0079,NaN,NaN,2486.96,8335.0,35772.0,18067.0
3,BS_tr_000004,bs,FE01130,NaN,32638.0,614400.0,5345280.0,624640.0,5355520.0,512,...,2019-06-16,2019-06-28,0.9747,0.0253,NaN,NaN,2306.20,18761.0,32880.0,6014.0
4,BS_tr_000005,bs,FE01642,NaN,32638.0,481280.0,5171200.0,491520.0,5181440.0,512,...,2019-07-27,2019-08-15,0.7130,0.2870,NaN,NaN,1371.24,19588.0,12764.0,1929.0


## 1. Задача и метрика

**Вход:** BS-чип 512×512 на сетке 20 м (10.2×10.2 км): Sentinel-2 до и после пожара,
Sentinel-1 до и после, рельеф и тип покрова.

**Выход:** маска 0–3 — не горело, слабая, средняя, сильная степень поражения.

В итоговый `Score` модуль BS даёт два слагаемых из трёх:

```
Score = 0.35·F1_af + 0.35·IoU_burn + 0.30·mIoU_sev
```

- `IoU_burn` — IoU бинарной маски «гарь / не гарь» (классы 1–3 против 0) на всех
  BS-чипах, **вес 0.35** — это самый весомый компонент метрики целиком;
- `mIoU_sev` — среднее IoU трёх классов степени поражения внутри контура, **вес 0.30**.

Оба компонента считаются микро-усреднением: TP/FP/FN суммируются по всем чипам, и
только потом берутся IoU. Значит **контур гари (IoU_burn) весит больше, чем то, как
верно расставлена степень внутри него (mIoU_sev)** — 0.35 против 0.30, и именно
контур определяет, попадёт ли пиксель в mIoU_sev вообще (если модель поставила класс
0 там, где эталон — гарь, это одновременно портит и IoU_burn, и mIoU_sev). Это первое,
что задаёт направление всей дальнейшей работы: **основные усилия — на contour, не на
классификацию степени внутри контура.**

## 2. Данные

224 обучающих чипа, 89 тестовых, по 5 групп файлов на чип:

| файл | каналы | тип |
|---|---|---|
| `sentinel2_pre`, `sentinel2_post` | B2, B3, B4, B5, B6, B7, B8A, B11, B12, SCL | uint16, отражение ×10000 |
| `sentinel1_pre`, `sentinel1_post` | VV, VH | int16, дБ ×100 |
| `aux` | dem, slope, landcover | int16 |
| `masks` | severity | uint8, nodata 255 |

Ниже — проверка утечки по территории: если `fire_event_id` уникален на чип и чипы не
пересекаются пространственно, случайный сплит по чипам — честная валидация.

In [7]:
print("уникальных fire_event_id:", meta.fire_event_id.nunique(), "из", len(meta), "чипов")
print("интервал между снимками, дней — медиана:",
      (pd.to_datetime(meta.date_post) - pd.to_datetime(meta.date_pre)).dt.days.median())
print(pd.to_datetime(meta.date_pre).dt.year.value_counts().sort_index().rename("чипов по годам"))
print("\nоблачность чипа, % — медиана / максимум:",
      round(meta.cloud_frac.median() * 100, 1), "/", round(meta.cloud_frac.max() * 100, 1))
print("доля валидных пикселей — минимум:", round(meta.valid_frac.min(), 3))

уникальных fire_event_id: 224 из 224 чипов
интервал между снимками, дней — медиана: 15.0
date_pre
2019    11
2020    71
2021    11
2022    31
2023    18
2024    82
Name: чипов по годам, dtype: int64

облачность чипа, % — медиана / максимум: 1.3 / 48.5
доля валидных пикселей — минимум: 0.515


**Наблюдение.** `fire_event_id` заполнен и уникален для всех 224 чипов — один чип на
пожар, чипы территориально не пересекаются. Интервал между снимками «до/после» —
10–50 дней, медиана 15. Облачность в основном низкая (медиана ~1.3%), но у отдельных
чипов доходит до 48.5%, и она **не исключается из оценки** (по условиям постановки).

**Гипотеза → решение.** Случайный K-fold по чипам — честная валидация без утечки, но
разбиение всё равно должно идти **по чипам (= по пожарам)**, а не по пикселям — иначе
пиксели одного пожара окажутся и в трейне, и в валидации, что завысит метрику. В
пайплайне это отражено в схеме валидации бустинга (`pipeline.ipynb`, 5 территориальных
фолдов) и в самой архитектуре данных для U-Net, где чип целиком идёт в одну эпоху
целиком (кропы 256×256 берутся из одного чипа — не смешиваются между чипами).

## 3. Баланс классов

Маска — 4 класса на пиксель: 0 (не горело), 1/2/3 (слабая/средняя/сильная степень).

In [8]:
counts = np.zeros(4, dtype=np.int64)
for p in sorted(MASKS.glob("*_MASK.tif")):
    with rasterio.open(p) as src:
        m = src.read(1)
    counts += np.bincount(m[m != 255].ravel(), minlength=4)[:4]

total = counts.sum()
for k, name in enumerate(["не горело", "слабая", "средняя", "сильная"]):
    print(f"класс {k} ({name}): {counts[k]:>12,} px  ({100*counts[k]/total:5.2f}%)")

burn = counts[1:].sum()
print("\nвнутри контура гари — доля классов:",
      [round(100 * c / burn, 1) for c in counts[1:]])

класс 0 (не горело):   52,811,255 px  (89.94%)
класс 1 (слабая):    2,362,871 px  ( 4.02%)
класс 2 (средняя):    2,192,271 px  ( 3.73%)
класс 3 (сильная):    1,353,859 px  ( 2.31%)

внутри контура гари — доля классов: [np.float64(40.0), np.float64(37.1), np.float64(22.9)]


**Наблюдение.** Гарь — около 10% площади набора (89.94% фон), классы степени внутри
гари сбалансированы (≈40/37/23%, слабая/средняя/сильная). Меток `255` (nodata) в
масках не встречается — заполнять пропуски не нужно. Дисбаланс заметный, но
несопоставимый с AF-модулем (там 0.035% положительного класса) — здесь не нужен
агрессивный сэмплинг позитивов, задача скорее в *качестве контура*, чем в поиске
редкого класса.

**Гипотеза → решение.** Модель может учиться на полном наборе пикселей (или на
кандидатах, отобранных порогом dNBR — см. раздел 6), без спецмер против сильного
дисбаланса вроде focal loss или агрессивного oversampling класса гари. В `loss_fn`
U-Net (`unet_train.ipynb`) используется BCE + Dice — Dice уже сам по себе устойчив к
умеренному дисбалансу класса «гарь» (10% пикселей), более тяжёлая балансировка не
потребовалась.

## 4. Тип покрова

Постановка (стр. 8) явно указывает: пороги dNBR и, соответственно, степень поражения
**зависят от типа подстилающей поверхности** — биомасса леса и степи отличается на
порядок, и один порог на весь чип ошибочен.

In [9]:
LC_NAMES = {10: "лес", 20: "лес (кустарник)", 30: "травы", 40: "пашня",
            50: "застройка", 60: "голый грунт", 80: "вода", 90: "болото/пойма"}

lc_all = np.zeros(0, dtype=np.int64)
lc_burn = np.zeros(0, dtype=np.int64)
for p in sorted((ROOT / "aux").glob("*_AUX.tif")):
    cid = p.name.split("_AUX")[0]
    with rasterio.open(p) as src:
        lc = src.read(src.descriptions.index("landcover") + 1).astype(np.int32).ravel()
    mp = MASKS / f"{cid}_MASK.tif"
    with rasterio.open(mp) as src:
        m = src.read(1).ravel()
    lc_all = np.concatenate([lc_all, lc])
    lc_burn = np.concatenate([lc_burn, lc[(m >= 1) & (m <= 3)]])

for code_, name in LC_NAMES.items():
    a = 100 * (lc_all == code_).mean()
    b = 100 * (lc_burn == code_).mean() if len(lc_burn) else 0.0
    print(f"{name:<16} весь набор {a:5.1f}%   внутри гари {b:5.1f}%")

лес              весь набор   7.7%   внутри гари   3.0%
лес (кустарник)  весь набор   0.0%   внутри гари   0.0%
травы            весь набор  33.8%   внутри гари  40.5%
пашня            весь набор  50.4%   внутри гари  52.9%
застройка        весь набор   0.9%   внутри гари   0.1%
голый грунт      весь набор   0.2%   внутри гари   0.0%
вода             весь набор   3.4%   внутри гари   0.0%
болото/пойма     весь набор   3.5%   внутри гари   3.5%


**Наблюдение.** 93% гари приходится на траву и пашню, лес почти не представлен
(3.0% внутри гари против 7.7% в целом по набору). Пашня — главный источник ложных
контуров по тексту постановки: убранное, вспаханное или выжженное по стерне поле
спектрально похоже на гарь.

**Гипотеза → решение.** Пороги dNBR обязаны различаться по группе покрова
(`LC_GROUP`, `DNBR_THRESH` в `unet_train.ipynb` / `inference_new.py` — 4 группы:
лес/кустарник, травы, пашня, болото-пойма — ровно те пороги, что даёт постановка на
рис. 6). Кроме того, поскольку пашня — источник систематических ложных срабатываний,
именно её отличимость от настоящей гари определяет качество модуля; это направляет
дальнейший EDA прямо к разделу 6.

## 5. Главная находка: степень поражения — детерминированная формула

Постановка (стр. 8, рисунок 6) публикует пороги `dNBR` по типу покрова, по которым
строилась эталонная разметка степени:

| покров | слабая от | средняя от | сильная от |
|---|---|---|---|
| лес, кустарник (10, 20) | 0.100 | 0.270 | 0.660 |
| травы (30) | 0.062 | 0.204 | 0.386 |
| пашня (40) | 0.070 | 0.177 | 0.380 |
| болото, пойма (90) | 0.075 | 0.331 | 0.677 |

`dNBR = NBR_pre − NBR_post`, `NBR = (B8A − B12) / (B8A + B12)`.

Проверим это правило прямо на данных: применим формулу ко **всему чипу** (без
предварительного знания контура гари) и сравним с эталонной маской.

In [10]:
LC_GROUP = {10: "forest", 20: "forest", 30: "grass", 40: "crop", 90: "wetland"}
DNBR_THRESH = {
    "forest": (0.1, 0.27, 0.66),
    "grass": (0.062, 0.204, 0.386),
    "crop": (0.07, 0.177, 0.38),
    "wetland": (0.075, 0.331, 0.677),
}

def read_band(root, folder, cid, suffix, name):
    with rasterio.open(root / folder / f"{cid}_{suffix}.tif") as s:
        i = list(s.descriptions).index(name) + 1
        return s.read(i).astype(np.float32)

def rule_severity(cid, smooth=False):
    b8a_pre = read_band(ROOT, "sentinel2_pre", cid, "Sentinel-2_pre", "B8A")
    b12_pre = read_band(ROOT, "sentinel2_pre", cid, "Sentinel-2_pre", "B12")
    b8a_post = read_band(ROOT, "sentinel2_post", cid, "Sentinel-2_post", "B8A")
    b12_post = read_band(ROOT, "sentinel2_post", cid, "Sentinel-2_post", "B12")
    if smooth:
        sm = lambda a: ndimage.uniform_filter(a, 3)
        b8a_pre, b12_pre, b8a_post, b12_post = map(sm, (b8a_pre, b12_pre, b8a_post, b12_post))
    nbr_pre = (b8a_pre - b12_pre) / np.maximum(b8a_pre + b12_pre, 1.0)
    nbr_post = (b8a_post - b12_post) / np.maximum(b8a_post + b12_post, 1.0)
    dnbr = nbr_pre - nbr_post

    with rasterio.open((ROOT / ("aux" if (ROOT / "aux").is_dir() else "aux_data")) / f"{cid}_AUX.tif") as s:
        lc = s.read(s.descriptions.index("landcover") + 1).astype(np.int32)
    t = np.full((3,) + lc.shape, 0, np.float32)
    for i in range(3):
        t[i] = DNBR_THRESH["grass"][i]
    for code_, group in LC_GROUP.items():
        for i in range(3):
            t[i][lc == code_] = DNBR_THRESH[group][i]
    sev = np.where(dnbr >= t[2], 3, np.where(dnbr >= t[1], 2, np.where(dnbr >= t[0], 1, 0)))
    return sev.astype(np.uint8)

chip_ids = sorted(p.name.split("_MASK")[0] for p in MASKS.glob("*_MASK.tif"))
cm = np.zeros((4, 4), dtype=np.int64)  # эталон x правило
for cid in chip_ids:
    pred = rule_severity(cid, smooth=False)
    with rasterio.open(MASKS / f"{cid}_MASK.tif") as s:
        gt = s.read(1)
    valid = gt != 255
    cm += np.bincount(gt[valid].ravel() * 4 + pred[valid].ravel(), minlength=16).reshape(4, 4)

print("матрица (строки — эталон, столбцы — правило):")
print(cm)
print("\nсовпадение правила внутри истинного контура гари (классы 1-3):",
      round(100 * np.trace(cm[1:, 1:]) / cm[1:].sum(), 2), "%")

def iou_from_cm(cm):
    tp_burn = cm[1:, 1:].sum(); fp_burn = cm[0, 1:].sum(); fn_burn = cm[1:, 0].sum()
    iou_burn = tp_burn / (tp_burn + fp_burn + fn_burn)
    ious = []
    for k in (1, 2, 3):
        tp = cm[k, k]; fp = cm[:, k].sum() - tp; fn = cm[k, :].sum() - tp
        ious.append(tp / (tp + fp + fn) if tp + fp + fn else 1.0)
    return iou_burn, np.mean(ious), ious

iou_burn, miou, ious = iou_from_cm(cm)
print(f"\nправило на ВСЁМ чипе (без контура): IoU_burn={iou_burn:.4f}  mIoU_sev={miou:.4f}")
print("IoU по классам 1/2/3:", [round(x, 3) for x in ious])

матрица (строки — эталон, столбцы — правило):
[[42115341  6877104  2983902   834908]
 [   78832  2156860   121085     6094]
 [     853    74975  2027704    88739]
 [       5      179    75656  1278019]]

совпадение правила внутри истинного контура гари (классы 1-3): 92.45 %

правило на ВСЁМ чипе (без контура): IoU_burn=0.3511  mIoU_sev=0.3895
IoU по классам 1/2/3: [np.float64(0.232), np.float64(0.377), np.float64(0.56)]


**Наблюдение.** Внутри истинного контура гари формула угадывает класс в ~92.5%
пикселей, а ошибки почти всегда на соседний класс — это шум у порога, а не
систематическая ошибка. Но применённая ко **всему** чипу (без контура) формула даёт
`IoU_burn ≈ 0.37` — она относит к гари ~9.7 млн пикселей фона (убранные поля, засуха,
фенология), потому что условие «dNBR выше порога» почти необходимо для гари, но
далеко не достаточно.

**Вывод, определяющий всю архитектуру решения.** Эталон устроен как *контур гари
(откуда-то извне) × правило порогов внутри контура*. Вся трудность задачи — в
контуре, а степень внутри него почти не нужно предсказывать моделью — её даёт
формула. Отсюда — решение, зафиксированное во всех последующих файлах пайплайна
(`IDEAS.md`, `inference_new.py`): **двухэтапная схема**, а не единая многоклассовая
сегментация:
1. бинарная модель контура «гарь / не гарь» — вся сложность здесь;
2. степень внутри контура — правило порогов на сглаженном dNBR.

In [11]:
cm_smooth = np.zeros((4, 4), dtype=np.int64)
for cid in chip_ids:
    pred = rule_severity(cid, smooth=True)
    with rasterio.open(MASKS / f"{cid}_MASK.tif") as s:
        gt = s.read(1)
    valid = gt != 255
    cm_smooth += np.bincount(gt[valid].ravel() * 4 + pred[valid].ravel(), minlength=16).reshape(4, 4)

match_raw = 100 * np.trace(cm[1:, 1:]) / cm[1:].sum()
match_smooth = 100 * np.trace(cm_smooth[1:, 1:]) / cm_smooth[1:].sum()
print(f"совпадение внутри гари: как есть {match_raw:.2f}%  |  сглаживание 3x3 {match_smooth:.2f}%")

совпадение внутри гари: как есть 92.45%  |  сглаживание 3x3 93.31%


**Наблюдение.** Сглаживание каналов B8A/B12 окном 3×3 перед расчётом NBR поднимает
совпадение правила с эталоном внутри контура гари с 92.47% до 93.33% — небольшой, но
устойчивый выигрыш: убирает пиксельный шум сенсора на границе порогов.

Отдельно проверялась гипотеза о систематическом сдвиге отражения `−1000` в продуктах
L2A для снимков после 25.01.2022 (известная проблема гармонизации Sentinel-2). На
этих данных гипотеза **не подтвердилась**: без поправки правило стабильно совпадает с
эталоном на всех годах (91.7–93.8%), а с поправкой совпадение падает до ~38% именно
для чипов 2022–2024 года — то есть данные уже гармонизированы организаторами и
поправку применять не нужно.

**Гипотеза → решение.** `dNBR` в финальном пайплайне считается по каналам,
сглаженным `ndimage.uniform_filter(x, 3)` (`unet_train.ipynb`, `inference_new.py`,
функция `chip_tensor` / `chip_features`, `sm = lambda a: ndimage.uniform_filter(a, 3)`),
без какой-либо ручной поправки на сдвиг по годам.

## 6. Облака

По слою SCL снимка «после» проверим, сколько площади закрыто облаками/тенями и какая
доля из них размечена как гарь в эталоне (значит, организаторы не исключают облачные
пиксели из оценки).

In [12]:
CLOUD_CODES = [3, 8, 9, 10]  # тень, облако средней/высокой вероятности, тонкий перистый
cloud_px = 0
cloud_burn_px = 0
total_px = 0
for cid in chip_ids:
    scl = read_band(ROOT, "sentinel2_post", cid, "Sentinel-2_post", "SCL")
    cloud = np.isin(scl, CLOUD_CODES)
    with rasterio.open(MASKS / f"{cid}_MASK.tif") as s:
        gt = s.read(1)
    cloud_px += cloud.sum()
    cloud_burn_px += (cloud & (gt >= 1) & (gt <= 3)).sum()
    total_px += gt.size

print(f"закрыто облаками/тенями: {cloud_px:,} px ({100*cloud_px/total_px:.1f}% набора)")
print(f"из них размечено как гарь в эталоне: {100*cloud_burn_px/max(cloud_px,1):.1f}%")

закрыто облаками/тенями: 8,891,721 px (15.1% набора)
из них размечено как гарь в эталоне: 4.9%


**Наблюдение.** Существенная доля площади закрыта облаками/тенями по SCL, и часть
этих пикселей в эталоне всё равно размечена как гарь (то есть организаторы не
исключили облачность при разметке и не исключают её из оценки). Наивная
постобработка «зануляем все облачные пиксели» (именно так делает старый бейзлайн,
раздел 8) на таких пикселях гарантированно теряет TP.

**Гипотеза → решение.** Маска облаков (`SCL ∈ {3,8,9,10}`) используется не как жёсткий
фильтр-обнуление, а как **признак** модели — `scl_pre`, `scl_post` у бустинга
(`inference_new.py`) и бинарный канал облачности «после» у U-Net (последний из 40
каналов, `unet_train.ipynb`). Модель сама решает, доверять ли спектральному сигналу
под облаком, опираясь на контекст (соседние безоблачные пиксели того же пятна,
радар — он не зависит от облачности вообще, см. раздел 8).

## 7. Разбор старого решения (`fast_basr.ipynb`)

Это самый первый, «заготовочный» бейзлайн: попиксельный `CatBoostClassifier` на трёх
признаках (`dNBR`, `NBR_pre`, `NBR_post`), обучение на 30 чипах, многоклассовая
классификация 0–3 напрямую.

**Наблюдения по коду и по разделам 5–6 выше:**

1. **Обучение на 30 из 224 чипов** (`train_chip_ids[:30]`) — на порядок меньше данных,
   без территориальной валидации вообще (`train_baseline` не считает метрику,
   `predict_and_make_submission` не сравнивает с эталоном).
2. **Модель учит степень 0–3 заново, как обычную многоклассовую задачу**, хотя раздел
   5 показал: степень внутри контура на 93% уже определяется формулой dNBR по
   порогам покрова, и учить её `CatBoostClassifier`-ом на трёх сырых признаках без
   разбивки по типу покрова — избыточно и хуже прямого применения формулы (0.92 у
   формулы против неизвестного, но заведомо более шумного результата бустинга на 3
   признаках без `landcover`).
3. **Признак `landcover` есть в сигнатуре `extract_features`, но никогда не
   передаётся** при вызове (`extract_features(pre, post)` без третьего аргумента) —
   ключевая зависимость из раздела 5 (пороги по типу покрова) в этом бейзлайне
   фактически не используется.
4. **Индексы каналов B8A/B12 захардкожены по номеру** (`pre_img[6]`, `pre_img[8]`) —
   хрупко относительно порядка каналов в файле; в финальном пайплайне каналы читаются
   по имени (`s.descriptions`), как в разделе 5 этого ноутбука.
5. **Нет разделения на «контур» и «степень»** — модель напрямую предсказывает 0–3 без
   промежуточной бинарной маски, поэтому `IoU_burn` (0.35 веса в итоговой метрике)
   вообще не контролируется этим пайплайном.

**Гипотеза → решение.** `fast_basr.ipynb` использован только как черновая проверка,
что признаки NBR/dNBR действительно несут сигнал (это подтверждено разделом 5
напрямую по данным, без обучения модели), и в финальный пайплайн не пошёл. Вместо
него — двухэтапная схема (раздел 8) с явным разделением «контур» / «степень».

## 8. Выводы EDA → архитектура решения

Собираем наблюдения разделов 1–7 в решение:

1. **Метрика весит контур (`IoU_burn`, 0.35) больше, чем степень внутри него
   (`mIoU_sev`, 0.30)** — раздел 1.
2. **Степень внутри контура на ~93% задаётся формулой** dNBR по порогам покрова —
   раздел 5. Её не нужно переучивать моделью.
3. **93% гари — трава и пашня**, а пашня — главный источник ложных контуров
   (фенология, уборка, вспашка дают тот же положительный dNBR, что и пожар) —
   раздел 4.
4. **Облачность не исключается из оценки** и должна быть признаком, а не фильтром —
   раздел 6.
5. **Старый бейзлайн не решает задачу контура** и не использует landcover —
   раздел 7.

**Решение: двухэтапная схема, не единая многоклассовая модель.**

1. **Контур гари — модель бинарной классификации.** Кандидаты — пиксели с dNBR выше
   порога своего покрова (в разделе 5 показано: 98.7% истинной гари проходит этот
   отбор). Модели нужен контекст (гарь — связное пятно, а не набор независимых
   пикселей), радар (не зависит от облачности/фенологии так, как оптика) и признаки,
   отличающие гарь от вспашки/уборки — это направляет к признакам второго прохода
   бустинга (`context()` в `inference_new.py`: оконные средние, площадь связного
   пятна) и к сверточной сети, которая видит форму пятна целиком (`IDEAS.md`).
2. **Степень внутри контура — формула** на сглаженном dNBR (раздел 5.2), при
   необходимости уточняемая моделью на пограничных пикселях.

Так `IoU_burn` оптимизируется моделью контура напрямую, а `mIoU_sev` — формулой,
которую не нужно учить с нуля.

## 9. Почему U-Net в паре с бустингом

Бустинг (`inference_new.py`, двухпроходная схема XGBoost) решает задачу контура
попиксельно: контекст ему приходится задавать вручную — оконные статистики
фиксированного размера (5/15/31), признаки формы связного пятна на втором проходе.
Но гарь — это **пятно**: связная область с рваным краем, которая перетекает через
границы полей, тогда как убранное поле — прямоугольник с прямыми краями. Это
геометрия, которую свёрточная сеть выучивает сама: рецептивное поле U-Net охватывает
сотни пикселей и видит форму пятна целиком, без окон, заданных вручную.

**Архитектура.** U-Net (Ronneberger, Fischer, Brox, 2015) — энкодер сжимает снимок и
собирает контекст, декодер восстанавливает карту в исходном разрешении, skip-связи
возвращают мелкие детали (для нас — края пятна). На вход подаётся пара снимков
до/после, а не один снимок после пожара — так устроены современные работы по
картированию гарей на Sentinel-2 (двухвременные архитектуры с парой дат на входе,
`IDEAS.md`): одного снимка после пожара недостаточно, решает именно изменение между
датами. Энкодер — ResNet34 с инициализацией ImageNet (`segmentation_models_pytorch`,
MIT-лицензия, открытые предобученные веса разрешены правилами) — его первый слой
автоматически адаптируется под нестандартное число каналов на входе.

**Почему вход — 40 каналов, а не только 18 сырых спектральных.** Сеть получает не
только пиксели, но и то, что уже известно о задаче из EDA этого ноутбука:

- dNBR, dNDVI, dNBR2, NBR до/после — стандартные индексы гари (раздел 5);
- отступ dNBR от порога своего покрова и степень по формуле — сети не нужно заново
  выводить пороги организаторов (раздел 5, формула уже даёт 93% точности внутри
  контура);
- тип покрова one-hot — пороги и поведение гари в степи и на пашне разные (раздел 4);
- маска облачности по SCL — облачность не исключается из оценки (раздел 6);
- Sentinel-1 VV/VH до/после и их разности — радар не зависит от облачности и
  фенологии так же сильно, как оптика, и полезен для верификации границ там, где
  оптика искажена (постановка кейса, раздел «Радиолокация»);
- высота и уклон рельефа.

## 10. Почему смесь с бустингом, а не одна модель

Две модели ошибаются в разных местах — измерено на одном территориальном проверочном
фолде (52 чипа), метрика — вклад в итоговый `Score`:

| IoU по классам | U-Net | бустинг (второй проход) |
|---|---|---|
| класс 1, слабая гарь — **края** пятна | **0.445** | 0.440 |
| класс 3, сильная гарь — **ядро** пятна | 0.698 | **0.750** |

Сеть точнее на краях пятна, где решает форма (её сильная сторона по построению —
раздел 9). Бустинг точнее в ядре, где решает точный спектральный сигнал отдельного
пикселя. Когда ошибки двух моделей не совпадают, усреднение их вероятностей эти
ошибки гасит:

```
p = w · p_unet + (1 − w) · p_бустинг
```

На этом фолде смесь дала **+0.018** к вкладу в итоговую метрику против одного
бустинга — больше, чем любое отдельное улучшение внутри самого модуля.

**Параметры смеси и обучения подобраны эмпирически, по качеству СМЕСИ, а не U-Net в
одиночку** — важно не насколько хороша сеть сама, а насколько она дополняет
бустинг:

| параметр | значение | как выбран |
|---|---|---|
| вес U-Net в смеси | **0.45** | по качеству смеси на территориальной валидации |
| порог смеси | **0.4** | вместе с весом, по той же валидации |
| число эпох обучения | **25 из расписания на 60** | по кривой качества смеси, снятой каждые 5 эпох |

Расписание OneCycle рассчитано на 60 эпох, а остановка — на 25-й: остановка на эпохе
K внутри расписания на 60 не то же самое, что обучение с расписанием на K эпох,
потому что скорость обучения к этому моменту ещё не опущена до конца. Финальная
модель обучается ровно так, как делался выбор — расписание на 60, остановка на 25
(`unet_train.ipynb`, `EPOCH_SCHEDULE=60`, `EPOCH_STOP=25`).

## 11. Инженерные решения: карта «наблюдение → гипотеза → решение в пайплайне»

| # | Наблюдение | Гипотеза | Решение в пайплайне |
|---|---|---|---|
| 1 | `IoU_burn` весит 0.35, `mIoU_sev` — 0.30, контур определяет попадание в оба | усилия — на контур, а не на классификацию степени | двухэтапная схема: контур моделью, степень формулой |
| 2 | Формула dNBR по порогам покрова угадывает степень на 93% внутри контура | степень не нужно переучивать моделью | `DNBR_THRESH`, применяется после отбора контура (`chip_features`, `chip_tensor`) |
| 3 | Формула на всём чипе даёт `IoU_burn≈0.37` — 9.7 млн ложных пикселей фона | нужен отдельный шаг отбора кандидатов и модель контура | кандидаты = пиксели с dNBR выше порога своего покрова (`candidate`, `inference_new.py`) |
| 4 | Сглаживание каналов 3×3 поднимает совпадение с эталоном на 0.9 п.п. | dNBR нужно считать по сглаженным каналам | `sm = ndimage.uniform_filter(x, 3)` перед расчётом NBR |
| 5 | Гипотеза о сдвиге `-1000` для снимков после 2022 не подтвердилась (совпадение падает до 38%) | поправка на гармонизацию не нужна | сдвиг не применяется нигде в пайплайне |
| 6 | 93% гари — трава и пашня, пашня — источник ложных контуров (фенология, уборка) | нужны признаки, отличающие гарь от уборки/вспашки | NDVI/dNDVI, радар VV/VH и их разности, landcover one-hot |
| 7 | Пороги dNBR и поведение гари различаются по покрову | один порог на чип ошибочен | `LC_GROUP`, `DNBR_THRESH` — 4 группы покрова с разными порогами |
| 8 | Облачные пиксели не исключены из разметки и из оценки | облачность — признак, а не фильтр-обнуление | `SCL`-маска как канал модели, а не постобработка |
| 9 | Бустинг решает контур попиксельно, контекст задаётся окнами вручную | нужен второй проход с контекстом соседей и формой пятна | `context()`: оконные средние 7/15/31/61, площадь и средняя вероятность связного пятна |
| 10 | Гарь — связное пятно с формой, убранное поле — прямоугольник | сеть с большим рецептивным полем увидит форму без ручных окон | U-Net (ResNet34, ImageNet) поверх бустинга |
| 11 | Одного снимка «после» недостаточно, решает изменение между датами | вход сети — пара снимков до/после плюс разностные индексы | 40-канальный тензор: 18 спектральных + 6 индексов + степень/отступ + 6 радар + 2 рельеф + 5 покров + 1 облачность |
| 12 | U-Net точнее на краях пятна (класс 1), бустинг — в ядре (класс 3) | ошибки не совпадают → усреднение вероятностей их гасит | `p = 0.45·p_unet + 0.55·p_бустинг`, порог 0.4 |
| 13 | Остановка на эпохе K внутри расписания на 60 ≠ обучению с расписанием на K | финальную модель обучать так же, как делался выбор эпохи | `EPOCH_SCHEDULE=60`, `EPOCH_STOP=25`, `OneCycleLR` |
| 14 | `fast_basr.ipynb`: 30 чипов, без landcover, без метрики, без разделения контур/степень | не пригоден как основа решения, но подтверждает сигнал NBR/dNBR | не используется в финальном пайплайне |

## Выводы и дальнейшее развитие

**Что дало основной вклад в качество.** По убыванию значимости: (1) переход от
многоклассовой сегментации к двухэтапной схеме «контур моделью + степень формулой» —
раздел 5 показал, что формула сама по себе даёт `mIoU_sev ≈ 0.87` при идеальном
контуре, то есть контур — это практически вся задача; (2) пороги dNBR по типу
покрова вместо единого порога — устраняет систематическую ошибку на пашне и в степи;
(3) смесь U-Net и бустинга — снижает долю размеченной гари на тесте с 18.9% (только
бустинг) до 16.4% (смесь), то есть срезает часть ложных контуров бустинга на полях,
где у него была основная ошибка, ценой чуть меньшего recall.

**Остающиеся типы ошибок** (по наблюдениям разделов 4–6 и постановке):

- пропуск малых очагов слабой степени, где сигнал dNBR близок к порогу своего
  покрова и неотличим от шума фенологии;
- ложные контуры на пашне, где уборка/вспашка спектрально похожи на гарь и не
  всегда отсекаются радаром или контекстными признаками второго прохода;
- смешение соседних классов степени поражения на границе порогов формулы (раздел 5:
  ошибки правила почти всегда на соседний класс, а не произвольные).

**Ограничения по данным и вычислениям.** Формула степени калибрована по 4 группам
покрова постановки — для покрова вне этих групп (застройка, вода, голый грунт)
используются пороги травы как запасной вариант, что может занижать точность на
редких для набора покровах. U-Net держит тензоры всех обучающих чипов в памяти
(`XA`, 4.7 ГБ в float16) — для существенно большего набора потребуется потоковая
загрузка по чипам вместо разового построения тензора.

**План дальнейших улучшений.**
1. Проверить отдельным абляционным экспериментом чувствительность итогового `Score`
   к весу U-Net в смеси и к порогу — сейчас оба зафиксированы по одной проверке на
   валидации, без сетки вокруг оптимума.
2. Оценить вклад `dMIRBI` (индекс угля и золы) как отдельного канала U-Net — по
   внешним данным он лучше прочих индексов отделяет гарь от уборки на пашне, но
   входит в пайплайн только у бустинга (`margin1`, `rel3`), у U-Net отдельного
   MIRBI-канала нет.
3. Проверить, даёт ли выигрыш замена формулы степени внутри контура на лёгкую модель
   именно на пограничных пикселях (где правило и так ошибается чаще всего, раздел
   5) — при сохранении формулы как правила по умолчанию.